# ⚡ ComfyUI Cloud Studio

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

### ✨ **Key Features:**
- 🚀 **Instant Canvas Boot:** Optimized to load your node graph and canvas in seconds without UI freeze or infinite loading.
- 📥 **Smart Model & Node Downloader:** Download Checkpoints, GGUF/UNets, LoRAs, VAEs, and custom nodes directly from Hugging Face and Civitai.
- 🧩 **ComfyUI-Manager Included:** Browse, install, and update your favorite custom nodes and workflows right inside the UI.
- 🌐 **5 Remote Access Options:** Connect easily through Cloudflare, Colab Native URL, Localtunnel, Pinggy, or Ngrok.
- 💾 **Google Drive Persistence & Workspace Backup:** Save your downloaded models, outputs, and custom nodes directly to Drive so you never lose your setup.
- 🔊 **Anti-Disconnect Keep-Alive:** Integrated browser activity guard to keep your Colab session from disconnecting while you create.

### 🔑 **Configuración de Seguridad y Credenciales**
Agrega tus tokens en Colab Secrets 🔑 o en los campos del formulario de abajo si descargas modelos privados o de Civitai.

In [ ]:
#@title 1. Initialize ComfyUI Environment (smart / cache-aware)
#@markdown Sentinela + detección GPU/torch + reinstalación condicional de deps.

import os
import subprocess
import sys
from google.colab import drive
from IPython.display import HTML, display

display(HTML('<div style="background: linear-gradient(90deg,#4b6cb7 0%,#182848 100%);color:#fff;padding:12px;border-radius:6px;font-weight:bold;text-align:center;">⚡ Inicializando ComfyUI...</div>'))

MOUNT_DRIVE = False #@param {type:"boolean"}
UPDATE_COMFY_UI = False #@param {type:"boolean"}
FORCE_REINSTALL = False #@param {type:"boolean"}

LOCAL_WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")
SENTINEL = os.path.join(LOCAL_WORKSPACE, ".setup_complete_v2")

def run(cmd, **kw):
    return subprocess.run(cmd, **kw)

def stream(cmd, cwd=None):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, cwd=cwd, bufsize=1)
    for line in iter(p.stdout.readline, ''):
        sys.stdout.write(line); sys.stdout.flush()
    p.wait()
    return p.returncode

# ---------- 1. Herramientas del sistema ----------
print("\n📦 [1/5] Verificando herramientas del sistema...")
if run(["which", "uv"], capture_output=True).returncode != 0:
    run(["pip", "install", "-q", "uv"])
if run(["which", "aria2c"], capture_output=True).returncode != 0 or \
   run(["which", "pigz"], capture_output=True).returncode != 0:
    run(["apt-get", "update", "-qq"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    run(["apt-get", "install", "-y", "-qq", "aria2", "pigz"],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if not os.path.exists("/usr/local/bin/cloudflared"):
    run(["wget", "-q", "-nc",
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
         "-O", "/usr/local/bin/cloudflared"])
    run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# ---------- 2. Drive ----------
if MOUNT_DRIVE:
    if not os.path.exists("/content/drive/MyDrive"):
        print("\n💾 Montando Google Drive...")
        drive.mount('/content/drive')

# ---------- 3. ComfyUI core ----------
print("\n📂 [2/5] Verificando ComfyUI core...")
if not os.path.exists(LOCAL_WORKSPACE):
    run(["git", "clone", "https://github.com/comfyanonymous/ComfyUI", LOCAL_WORKSPACE])
elif UPDATE_COMFY_UI:
    print("   🔄 git pull...")
    run(["git", "pull"], cwd=LOCAL_WORKSPACE)

if not os.path.exists(os.path.join(LOCAL_WORKSPACE, "main.py")):
    raise SystemExit("❌ ComfyUI core no quedó instalado (falta main.py).")

# ---------- 4. Symlinks a Drive + restore tar ----------
if MOUNT_DRIVE:
    for d in ["models", "output", "input"]:
        local_path = os.path.join(LOCAL_WORKSPACE, d)
        drive_path = os.path.join(DRIVE_WORKSPACE, d)
        os.makedirs(drive_path, exist_ok=True)
        if os.path.exists(local_path) and not os.path.islink(local_path):
            import shutil as _sh; _sh.rmtree(local_path)
        if not os.path.exists(local_path):
            os.symlink(drive_path, local_path)
    if os.path.exists(CACHE_TAR):
        print("   📦 Restaurando workspace cacheado...")
        os.system(f"tar -I pigz -xf '{CACHE_TAR}' -C '{LOCAL_WORKSPACE}' >/dev/null 2>&1")

# ---------- 5. Estructura user + Manager ----------
os.makedirs(os.path.join(LOCAL_WORKSPACE, "custom_nodes"), exist_ok=True)
ud = os.path.join(LOCAL_WORKSPACE, "user", "default")
for sub in ["subgraphs", "workflows"]:
    os.makedirs(os.path.join(ud, sub), exist_ok=True)
open(os.path.join(ud, "user.css"), "a").close()
tp = os.path.join(ud, "comfy.templates.json")
if not os.path.exists(tp):
    open(tp, "w").write("{}")

manager_path = os.path.join(LOCAL_WORKSPACE, "custom_nodes", "ComfyUI-Manager")
if not os.path.exists(os.path.join(manager_path, "__init__.py")):
    if os.path.exists(manager_path):
        import shutil as _sh; _sh.rmtree(manager_path)
    print("\n📦 [3/5] Instalando ComfyUI-Manager...")
    run(["git", "clone", "https://github.com/Comfy-Org/ComfyUI-Manager.git", manager_path])

import shutil as _sh
_sh.rmtree(os.path.join(LOCAL_WORKSPACE, "user", "__manager", ".legacy-manager-backup"),
           ignore_errors=True)

mgr_cfg = """[default]
network_mode = standard
preview_method = auto
badge_mode = None
security_level = normal
migrated = true
skip_update_check = true
component_policy = remote
lazy_load = true
"""
for d in [os.path.join(LOCAL_WORKSPACE, "user", "default", "ComfyUI-Manager"),
          os.path.join(LOCAL_WORKSPACE, "user", "ComfyUI-Manager"),
          os.path.join(LOCAL_WORKSPACE, "user", "__manager"),
          manager_path]:
    os.makedirs(d, exist_ok=True)
    with open(os.path.join(d, "config.ini"), "w") as f:
        f.write(mgr_cfg)

# ---------- 6. Dependencias Python (solo si sentinel no existe) ----------
ALREADY = (not FORCE_REINSTALL) and os.path.exists(SENTINEL) and \
          os.path.exists(os.path.join(LOCAL_WORKSPACE, "main.py"))

if ALREADY:
    print("\n🛠️ [4/5] Deps Python ya instaladas (sentinel). Se omite reinstalación.")
else:
    print("\n🛠️ [4/5] Instalando dependencias vía UV...")
    stream(["uv", "pip", "install", "--system", "-r", "requirements.txt"], cwd=LOCAL_WORKSPACE)
    stream(["uv", "pip", "install", "--system",
            "av", "imageio-ffmpeg", "torchsde", "einops", "transformers", "safetensors",
            "kornia", "spandrel", "scipy", "soundfile", "insightface", "onnxruntime-gpu",
            "alembic", "blake3", "comfy-kitchen", "comfy-aimdo", "comfy-angle", "gguf", "piexif",
            "comfyui-frontend-package", "pydantic", "pydantic-settings", "simpleeval", "accelerate"])
    mgr_req = os.path.join(manager_path, "requirements.txt")
    if os.path.exists(mgr_req):
        stream(["uv", "pip", "install", "--system", "-r", mgr_req])
    open(SENTINEL, "w").write("ok")

# ---------- 7. Diagnóstico GPU / torch ----------
print("\n🩺 [5/5] Diagnóstico de aceleración...")
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"   torch={torch.__version__} | CUDA compiled={torch.version.cuda} | available={cuda_ok}")
    if cuda_ok:
        print(f"   ✅ GPU: {torch.cuda.get_device_name(0)}")
    else:
        print("   ⚠️ CUDA NO disponible. Cambia a Runtime→Change runtime type→T4 GPU.")
        print("   ⚠️ Si el runtime ya es GPU, reinstala torch CUDA manualmente:")
        print("      uv pip install --system torch torchvision torchaudio \\")
        print("        --index-url https://download.pytorch.org/whl/cu121")
except Exception as e:
    print(f"   ❌ torch no importable: {e}")

print("\n✅ [SYSTEM READY] ComfyUI inicializado.")

In [ ]:
#@title 2. Workflow Presets Catalog & Smart Downloader
#@markdown Selecciona un **Workflow Preset** para autodescargar el modelo, text encoder, VAE y custom nodes necesarios en un solo clic, o añade tus propios links en el formulario.

#@markdown ### ⚡ Selección de Workflow Preset
WORKFLOW_PRESET = "Anima Kirazuri v4 - Anime (Int8 / T4 Safe)" #@param ["Anima Kirazuri v4 - Anime (Int8 / T4 Safe)", "Anima Kirazuri v4 - Anime (BF16 Full Quality)", "Qwen-Image-2.1 (Full Setup)", "Personalizado (Solo URLs manuales)"]
ACCION = "Descargar e Instalar Todo (1 Clic)" #@param ["Descargar e Instalar Todo (1 Clic)", "Solo Poblar la Cola", "Descargar Cola Pendiente", "🧹 Purgar Residuos y Sanar Caché"]

#@markdown ---
#@markdown ### 🛡️ Mantenimiento y Auto-Purga de Residuos
AUTO_PURGA_RESIDUOS = True #@param {type:"boolean"}
FORCE_REDOWNLOAD    = False #@param {type:"boolean"}

#@markdown ---
#@markdown ### 🔑 Credenciales (Opcional - solo para modelos privados o Civitai)
HF_TOKEN = "" #@param {type:"string"}
CIVITAI_API_KEY = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 📦 URLs Personalizadas / Adicionales (Pega links separados por coma o salto de línea)
DIFFUSION_URLS        = "" #@param {type:"string"}
TEXT_ENCODER_URLS     = "" #@param {type:"string"}
CHECKPOINT_URLS       = "" #@param {type:"string"}
UNET_DIFFUSION_URLS   = "" #@param {type:"string"}
VAE_URLS              = "" #@param {type:"string"}
LORA_URLS             = "" #@param {type:"string"}
CONTROLNET_URLS       = "" #@param {type:"string"}
UPSCALE_MODELS_URLS   = "" #@param {type:"string"}
CUSTOM_NODE_URLS      = "" #@param {type:"string"}

#@markdown ---
#@markdown ### 💾 Guardar como nuevo Preset en Google Drive (Opcional)
GUARDAR_COMO_PRESET = "" #@param {type:"string"}

import os
import re
import json
import time
import subprocess
import urllib.parse
import requests
from IPython.display import display, clear_output, HTML

WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
PRESETS_FILE = os.path.join(DRIVE_WORKSPACE, "presets.json") if os.path.exists("/content/drive/MyDrive") else os.path.join(WORKSPACE, "presets.json")

# =========================================================
#  1. RESOLUCIÓN DE CREDENCIALES (SECRETS / FORMULARIO)
# =========================================================
def _get_colab_secret(secret_name):
    try:
        from google.colab import userdata
        val = userdata.get(secret_name)
        return (val or "").strip()
    except Exception:
        return ""

if not HF_TOKEN.strip():
    HF_TOKEN = _get_colab_secret("HF_TOKEN")

if not CIVITAI_API_KEY.strip():
    CIVITAI_API_KEY = _get_colab_secret("CIVITAI_API_KEY")

# =========================================================
#  2. MAPEO DE DIRECTORIOS
# =========================================================
DIRS = {
    "diffusion_models/anima": os.path.join(WORKSPACE, "models/diffusion_models/anima"),
    "diffusion_models/qwen":  os.path.join(WORKSPACE, "models/diffusion_models/qwen"),
    "diffusion_models":       os.path.join(WORKSPACE, "models/diffusion_models"),
    "text_encoders":          os.path.join(WORKSPACE, "models/text_encoders"),
    "checkpoints":            os.path.join(WORKSPACE, "models/checkpoints"),
    "unet":                   os.path.join(WORKSPACE, "models/unet"),
    "clip":                   os.path.join(WORKSPACE, "models/clip"),
    "clip_vision":            os.path.join(WORKSPACE, "models/clip_vision"),
    "vae":                    os.path.join(WORKSPACE, "models/vae"),
    "loras":                  os.path.join(WORKSPACE, "models/loras"),
    "controlnet":             os.path.join(WORKSPACE, "models/controlnet"),
    "upscale_models":         os.path.join(WORKSPACE, "models/upscale_models"),
    "latent_upscale_models":  os.path.join(WORKSPACE, "models/latent_upscale_models"),
    "embeddings":             os.path.join(WORKSPACE, "models/embeddings"),
    "custom_nodes":           os.path.join(WORKSPACE, "custom_nodes"),
}

VALID_EXTS = ('.safetensors', '.ckpt', '.pt', '.pth', '.bin', '.gguf', '.sft', '.pkl')
MIN_VALID_SIZE = 1024 * 1024
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"

# =========================================================
#  3. FUNCIÓN DE HIGIENIZACIÓN Y REDIRECCIÓN DE URLS
# =========================================================
def clean_url(url):
    url = (url or "").strip()
    if not url:
        return ""
    if "huggingface.co" in url and "/blob/" in url:
        url = url.replace("/blob/", "/resolve/")
    # Redirección forzada de punteros LFS rotos de Kirazuri Anima hacia sus binarios oficiales
    if "motimalu/kirazuri-anima" in url:
        if "text_encoders/qwen_3_06b_base.safetensors" in url or "qwen_3_06b_base.safetensors" in url:
            return "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
        if "vae/qwen_image_vae.safetensors" in url or "qwen_image_vae.safetensors" in url:
            return "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
    return url

# =========================================================
#  4. CATÁLOGO DE PRESETS (JSON PERSISTENTE)
# =========================================================
DEFAULT_PRESETS = {
    "Anima Kirazuri v4 - Anime (Int8 / T4 Safe)": {
        "title": "🌸 Anima Kirazuri v4 - Anime (Int8 / T4 Safe)",
        "description": "Kirazuri Anima v4 int8 cuantizado + Qwen 3 0.6B Base + Qwen Image VAE + Turbo LoRA + Prompt Control.",
        "models": {
            "diffusion_models/anima": [
                "https://huggingface.co/motimalu/kirazuri-anima/resolve/main/split_files/diffusion_models/anima-kirazuri-v4-int8-convrot.safetensors"
            ],
            "text_encoders": [
                "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
            ],
            "vae": [
                "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
            ],
            "loras": [
                "https://huggingface.co/motimalu/kirazuri-anima/resolve/main/split_files/diffusion_models/kirazuri-anima-turbo-v4.safetensors"
            ]
        },
        "custom_nodes": [
            "https://github.com/asagi4/comfyui-prompt-control",
            "https://github.com/newtextdoc1111/ComfyUI-Autocomplete-Plus"
        ]
    },
    "Anima Kirazuri v4 - Anime (BF16 Full Quality)": {
        "title": "🌸 Anima Kirazuri v4 - Anime (BF16 Full Quality)",
        "description": "Kirazuri Anima v4 BF16 nativo (3.89 GB) + Qwen 3 0.6B Base + Qwen Image VAE + Turbo LoRA + Prompt Control.",
        "models": {
            "diffusion_models/anima": [
                "https://huggingface.co/motimalu/kirazuri-anima/resolve/main/split_files/diffusion_models/anima-kirazuri-v4.safetensors"
            ],
            "text_encoders": [
                "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors"
            ],
            "vae": [
                "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors"
            ],
            "loras": [
                "https://huggingface.co/motimalu/kirazuri-anima/resolve/main/split_files/diffusion_models/kirazuri-anima-turbo-v4.safetensors"
            ]
        },
        "custom_nodes": [
            "https://github.com/asagi4/comfyui-prompt-control",
            "https://github.com/newtextdoc1111/ComfyUI-Autocomplete-Plus"
        ]
    },
    "Qwen-Image-2.1 (Full Setup)": {
        "title": "⚡ Qwen-Image-2.1 (Full Setup)",
        "description": "Workflow optimizado para Qwen-Image-2.1 (diffusion_models/qwen, text_encoders, VAE bf16, KJNodes y DeGrid).",
        "models": {
            "diffusion_models/qwen": [
                "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/diffusion_models/qwen_image_2.1_int8_convrot.safetensors"
            ],
            "text_encoders": [
                "https://huggingface.co/pottokao/Qwen-Image-2.1-Text-Encoder-Heretic-W4A8/resolve/main/qwen3vl_8b_w4a8_heretic.safetensors"
            ],
            "vae": [
                "https://huggingface.co/Comfy-Org/Qwen-Image-2.1/resolve/main/vae/qwen_image_2.1_vae_bf16.safetensors"
            ]
        },
        "custom_nodes": [
            "https://github.com/kijai/ComfyUI-KJNodes",
            "https://github.com/lunaaispace-eng/ComfyUI-DeGrid"
        ]
    }
}

def load_presets_catalog():
    user_presets = {}
    if os.path.exists(PRESETS_FILE):
        try:
            with open(PRESETS_FILE, "r", encoding="utf-8") as f:
                user_presets = json.load(f)
        except Exception:
            pass

    # Higienizar URLs en presets antiguos almacenados en disco
    for p_name, p_data in user_presets.items():
        if isinstance(p_data, dict) and "models" in p_data:
            for cat, urls in p_data["models"].items():
                p_data["models"][cat] = [clean_url(u) for u in urls]

    # Los DEFAULT_PRESETS oficiales actualizados sobreescriben presets previos con errores
    merged = {**user_presets, **DEFAULT_PRESETS}
    try:
        os.makedirs(os.path.dirname(PRESETS_FILE), exist_ok=True)
        with open(PRESETS_FILE, "w", encoding="utf-8") as f:
            json.dump(merged, f, indent=2, ensure_ascii=False)
    except Exception:
        pass
    return merged

CATALOG = load_presets_catalog()

# =========================================================
#  5. MANIFEST (IDEMPOTENCIA DE DESCARGAS)
# =========================================================
DRIVE_MANIFEST = "/content/drive/MyDrive/ComfyUI/.download_manifest.json"
LOCAL_MANIFEST = os.path.join(WORKSPACE, ".download_manifest.json")
MANIFEST_PATH = DRIVE_MANIFEST if os.path.exists("/content/drive/MyDrive") else LOCAL_MANIFEST

def load_manifest():
    if os.path.exists(MANIFEST_PATH):
        try:
            with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
                return json.load(f)
        except Exception:
            pass
    return {}

def save_manifest(m):
    try:
        os.makedirs(os.path.dirname(MANIFEST_PATH), exist_ok=True)
        tmp = MANIFEST_PATH + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(m, f, indent=2)
        os.replace(tmp, MANIFEST_PATH)
    except Exception:
        pass

MANIFEST = load_manifest()

# =========================================================
#  6. SISTEMA DE AUTO-PURGA Y SANACIÓN DE CACHÉ
# =========================================================
def purgar_residuos_y_sanar_cache(verbose=True):
    purged_items = []
    models_root = os.path.join(WORKSPACE, "models")
    search_dirs = [models_root] + [d for d in DIRS.values() if os.path.exists(d)]
    
    # 1. Purgar archivos de control de aria2, temporales y punteros LFS corruptos
    scanned_paths = set()
    for base in search_dirs:
        if not os.path.exists(base):
            continue
        for root, _, files in os.walk(base):
            for fn in files:
                p = os.path.join(root, fn)
                if p in scanned_paths:
                    continue
                scanned_paths.add(p)
                
                # Archivos residuales de descargas interrumpidas
                if fn.endswith(('.aria2', '.tmp', '.part', '.crdownload')):
                    try:
                        os.remove(p)
                        purged_items.append(f"Archivo temporal/aria2: {fn}")
                    except Exception:
                        pass
                elif any(fn.lower().endswith(ext) for ext in VALID_EXTS):
                    if os.path.islink(p):
                        continue
                    try:
                        sz = os.path.getsize(p)
                        if sz < MIN_VALID_SIZE:
                            os.remove(p)
                            purged_items.append(f"Binario corrupto o puntero Git-LFS ({sz}B): {fn}")
                    except Exception:
                        pass

    # 2. Purgar enlaces simbólicos rotos (dangling symlinks)
    for base in search_dirs:
        if not os.path.exists(base):
            continue
        for fn in os.listdir(base):
            p = os.path.join(base, fn)
            if os.path.islink(p) and not os.path.exists(p):
                try:
                    os.unlink(p)
                    purged_items.append(f"Enlace roto (symlink): {fn}")
                except Exception:
                    pass

    # 3. Sanar Manifiesto de descargas (eliminar registros de archivos inexistentes o < 1MB)
    global MANIFEST
    keys_to_delete = []
    for k, v in list(MANIFEST.items()):
        target = os.path.join(v.get("dir", ""), v.get("filename", ""))
        if not os.path.exists(target) or os.path.getsize(target) < MIN_VALID_SIZE:
            keys_to_delete.append(k)
            if k in MANIFEST:
                del MANIFEST[k]

    if keys_to_delete:
        save_manifest(MANIFEST)
        purged_items.append(f"{len(keys_to_delete)} entrada(s) obsoleta(s) eliminada(s) del Manifiesto.")

    # 4. Sanar Cola de Descargas (.download_queue.json)
    queue_file = ("/content/drive/MyDrive/ComfyUI/.download_queue.json"
                  if os.path.exists("/content/drive/MyDrive")
                  else os.path.join(WORKSPACE, ".download_queue.json"))
    if os.path.exists(queue_file):
        try:
            with open(queue_file, "r", encoding="utf-8") as f:
                saved_q = json.load(f)
            updated_q = {}
            for cat, urls in saved_q.items():
                updated_q[cat] = [clean_url(u) for u in urls if clean_url(u)]
            with open(queue_file, "w", encoding="utf-8") as f:
                json.dump(updated_q, f, indent=2)
            purged_items.append("Cola .download_queue.json saneada e higienizada.")
        except Exception:
            pass

    if verbose:
        if purged_items:
            print("🧹 [Auto-Purga] Se limpiaron los siguientes residuos del entorno:")
            for item in purged_items:
                print(f"   ↳ 🗑️ {item}")
        else:
            print("✨ [Auto-Purga] El entorno y la caché están limpios de residuos.")
    return len(purged_items)

# =========================================================
#  7. HELPERS DE RED Y NOMBRES DE ARCHIVOS
# =========================================================
def normalize_url_key(url):
    u = re.sub(r'[?&]token=[^&]+', '', url).rstrip('?&')
    m = re.search(r'civitai\.(?:com|red)/api/download/models/(\d+)', u)
    if m:
        fm = re.search(r'fileId=(\d+)', u)
        return f"civitai:{m.group(1)}:{fm.group(1) if fm else 'default'}"
    return u

def mask(url):
    return re.sub(r'([?&]token=)[^&]+', r'\1***', url or "")

def build_headers(url):
    h = {"User-Agent": UA}
    if 'civitai' in url:
        h['Referer'] = 'https://civitai.com/'
        if CIVITAI_API_KEY.strip():
            h['Authorization'] = f'Bearer {CIVITAI_API_KEY.strip()}'
    if 'huggingface.co' in url and HF_TOKEN.strip():
        h['Authorization'] = f'Bearer {HF_TOKEN.strip()}'
    return h

def parse_civitai_target(url, civitai_key=""):
    version_id = None
    direct_url = url
    if "civitai.com" in url or "civitai.red" in url:
        m_dl = re.search(r'civitai\.(?:com|red)/api/download/models/(\d+)', url)
        if m_dl:
            version_id = m_dl.group(1)
        else:
            parsed = urllib.parse.urlparse(url)
            qs = urllib.parse.parse_qs(parsed.query)
            if 'modelVersionId' in qs:
                version_id = qs['modelVersionId'][0]
                file_param = f"?fileId={qs['fileId'][0]}" if 'fileId' in qs else ''
                direct_url = f"https://civitai.com/api/download/models/{version_id}{file_param}"
            else:
                m_model = re.search(r'/models/(\d+)', parsed.path)
                if m_model:
                    model_id = m_model.group(1)
                    try:
                        headers = {"User-Agent": UA}
                        if civitai_key:
                            headers["Authorization"] = f"Bearer {civitai_key}"
                        r = requests.get(f"https://civitai.com/api/v1/models/{model_id}", headers=headers, timeout=12)
                        if r.status_code == 200:
                            data = r.json()
                            versions = data.get("modelVersions", [])
                            if versions:
                                version_id = str(versions[0]["id"])
                                direct_url = f"https://civitai.com/api/download/models/{version_id}"
                    except Exception as e:
                        print(f"      ⚠️ Civitai Model API info error: {e}")

        if civitai_key and "token=" not in direct_url:
            sep = "&" if "?" in direct_url else "?"
            direct_url = f"{direct_url}{sep}token={civitai_key}"

    return direct_url, version_id

def extract_filename(headers, final_url):
    cd = headers.get('content-disposition', '')
    if cd:
        m = re.findall(r'filename\*?=(?:UTF-8\'\')?"?([^;\r\n"]+)"?', cd)
        if m:
            return urllib.parse.unquote(m[0].strip())
    path = urllib.parse.urlparse(final_url).path
    return os.path.basename(path) or None

def filename_from_civitai_api(version_id, civitai_key=""):
    if not version_id:
        return None
    try:
        headers = {"User-Agent": UA}
        if civitai_key:
            headers["Authorization"] = f"Bearer {civitai_key}"
        r = requests.get(f"https://civitai.com/api/v1/model-versions/{version_id}", headers=headers, timeout=15)
        if r.status_code == 200:
            files = r.json().get('files', [])
            if files:
                primary = next((f for f in files if f.get('primary')), files[0])
                name = primary.get('name')
                if name:
                    return name
    except Exception as e:
        print(f"      ⚠️ Civitai Version API: {e}")
    return None

def resolve_filename(url, civitai_ver_id=None):
    if civitai_ver_id:
        fn = filename_from_civitai_api(civitai_ver_id, CIVITAI_API_KEY)
        if fn and any(fn.lower().endswith(ext) for ext in VALID_EXTS):
            print(f"      ✅ Nombre vía Civitai API: {fn}")
            return fn

    path = urllib.parse.urlparse(url).path
    base = os.path.basename(path)
    if base and any(base.lower().endswith(ext) for ext in VALID_EXTS):
        fn = urllib.parse.unquote(base)
        print(f"      ✅ Nombre extraído de URL: {fn}")
        return fn

    headers = build_headers(url)
    try:
        r = requests.head(url, headers=headers, allow_redirects=True, timeout=12)
        if r.status_code in (200, 302):
            fn = extract_filename(r.headers, r.url)
            if fn and any(fn.lower().endswith(ext) for ext in VALID_EXTS):
                print(f"      ✅ Nombre vía HEAD: {fn}")
                return fn
    except Exception:
        pass

    try:
        with requests.get(url, headers=headers, allow_redirects=True, stream=True, timeout=12) as r:
            if r.status_code == 200:
                fn = extract_filename(r.headers, r.url)
                if fn and any(fn.lower().endswith(ext) for ext in VALID_EXTS):
                    print(f"      ✅ Nombre vía GET stream: {fn}")
                    return fn
    except Exception:
        pass

    if base:
        return urllib.parse.unquote(base)
    return "downloaded_model.safetensors"

def _create_helper_symlinks(target_dir, filename, out):
    # Enlaza subcarpetas específicas (qwen, anima) hacia la raíz diffusion_models/
    if target_dir.endswith("diffusion_models/anima") or target_dir.endswith("diffusion_models/qwen"):
        parent_dir = os.path.dirname(target_dir)
        parent_sym = os.path.join(parent_dir, filename)
        if not os.path.exists(parent_sym):
            try:
                os.symlink(out, parent_sym)
            except Exception:
                pass
    elif target_dir.endswith("diffusion_models"):
        if "qwen" in filename.lower():
            qwen_dir = os.path.join(target_dir, "qwen")
            os.makedirs(qwen_dir, exist_ok=True)
            qwen_sym = os.path.join(qwen_dir, filename)
            if not os.path.exists(qwen_sym):
                try:
                    os.symlink(out, qwen_sym)
                except Exception:
                    pass
        if "anima" in filename.lower() or "kirazuri" in filename.lower():
            anima_dir = os.path.join(target_dir, "anima")
            os.makedirs(anima_dir, exist_ok=True)
            anima_sym = os.path.join(anima_dir, filename)
            if not os.path.exists(anima_sym):
                try:
                    os.symlink(out, anima_sym)
                except Exception:
                    pass

    # Enlaces cruzados entre text_encoders y clip
    if target_dir.endswith("text_encoders"):
        clip_dir = os.path.join(WORKSPACE, "models/clip")
        os.makedirs(clip_dir, exist_ok=True)
        clip_sym = os.path.join(clip_dir, filename)
        if not os.path.exists(clip_sym):
            try:
                os.symlink(out, clip_sym)
            except Exception:
                pass
    elif target_dir.endswith("models/clip"):
        te_dir = os.path.join(WORKSPACE, "models/text_encoders")
        os.makedirs(te_dir, exist_ok=True)
        te_sym = os.path.join(te_dir, filename)
        if not os.path.exists(te_sym):
            try:
                os.symlink(out, te_sym)
            except Exception:
                pass

def download_aria2(url, filename, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    cmd = ["aria2c", f"--user-agent={UA}", "--continue=true",
           "--max-connection-per-server=16", "--split=16", "--min-split-size=1M",
           "--file-allocation=none", "--max-tries=3", "--retry-wait=3",
           "--summary-interval=0", "--console-log-level=warn",
           f"--dir={target_dir}", f"--out={filename}"]
    for k, v in build_headers(url).items():
        cmd.append(f"--header={k}: {v}")
    cmd.append(url)
    print(f"      🚀 aria2c (16 hilos)...")
    res = subprocess.run(cmd, capture_output=True, text=True)
    out = os.path.join(target_dir, filename)
    if res.returncode == 0 and os.path.exists(out):
        sz = os.path.getsize(out)
        if sz >= MIN_VALID_SIZE:
            print(f"      ✅ aria2c OK ({sz/1e6:.1f} MB)")
            return True
        else:
            try:
                with open(out, 'r', errors='ignore') as f:
                    first_line = f.readline()
                if "git-lfs" in first_line:
                    print(f"      ⚠️ Advertencia: El archivo descargado es solo un puntero Git-LFS ({sz} bytes).")
            except Exception:
                pass
            os.remove(out)
    print(f"      ❌ aria2c exit={res.returncode}")
    return False

def download_requests(url, filename, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    print("      🔁 Fallback requests...")
    try:
        with requests.get(url, headers=build_headers(url), allow_redirects=True, stream=True, timeout=60) as r:
            r.raise_for_status()
            real = extract_filename(r.headers, r.url) or filename
            out = os.path.join(target_dir, real)
            total = int(r.headers.get('content-length', 0))
            done = 0
            with open(out, 'wb') as f:
                for ch in r.iter_content(1024*1024):
                    if ch:
                        f.write(ch)
                        done += len(ch)
                        if total:
                            print(f"\r      ⬇️ {done*100/total:5.1f}%", end="")
            print()
            if total and done != total:
                return False
            sz = os.path.getsize(out)
            if sz < MIN_VALID_SIZE:
                try:
                    with open(out, 'r', errors='ignore') as f:
                        first_line = f.readline()
                    if "git-lfs" in first_line:
                        print(f"      ⚠️ El archivo descargado es solo un puntero Git-LFS ({sz} bytes) y no el binario real.")
                except Exception:
                    pass
                os.remove(out)
                return False
            return True
    except Exception as e:
        print(f"\n      ❌ requests: {e}")
        return False

def download_file(raw_url, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    cleaned = clean_url(raw_url)
    url, civitai_ver = parse_civitai_target(cleaned, CIVITAI_API_KEY)
    key = normalize_url_key(url)
    entry = MANIFEST.get(key)

    if entry and not FORCE_REDOWNLOAD:
        full = os.path.join(entry["dir"], entry["filename"])
        if os.path.exists(full):
            actual, expected = os.path.getsize(full), entry.get("size", 0)
            if not expected or actual == expected:
                print(f"   ⏭️ SKIP (manifest): {entry['filename']} ({actual/1e6:.1f} MB)")
                _create_helper_symlinks(target_dir, entry['filename'], full)
                return True

    print(f"   📥 Descargando: {mask(url)}")
    filename = resolve_filename(url, civitai_ver)
    out = os.path.join(target_dir, filename)

    if os.path.exists(out) and not FORCE_REDOWNLOAD and os.path.getsize(out) >= MIN_VALID_SIZE:
        size = os.path.getsize(out)
        print(f"   ⏭️ SKIP (archivo ya existe): {filename} ({size/1e6:.1f} MB)")
        MANIFEST[key] = {"url": mask(url), "filename": filename, "dir": target_dir, "size": size, "ts": time.time()}
        save_manifest(MANIFEST)
        _create_helper_symlinks(target_dir, filename, out)
        return True

    ok = download_aria2(url, filename, target_dir) or download_requests(url, filename, target_dir)
    if not ok:
        print(f"   ❌ FALLÓ: {mask(url)}")
        return False

    if not os.path.exists(out):
        cands = [f for f in os.listdir(target_dir) if f.lower().endswith(VALID_EXTS)]
        if cands:
            out = os.path.join(target_dir, cands[0])
            filename = os.path.basename(out)

    size = os.path.getsize(out) if os.path.exists(out) else 0
    MANIFEST[key] = {"url": mask(url), "filename": filename, "dir": target_dir, "size": size, "ts": time.time()}
    save_manifest(MANIFEST)
    print(f"   💾 OK: {filename} ({size/1e6:.1f} MB)")
    _create_helper_symlinks(target_dir, filename, out)
    return True

def download_node(raw_url, target_dir):
    os.makedirs(target_dir, exist_ok=True)
    node = raw_url.rstrip('/').split('/')[-1].replace('.git', '')
    path = os.path.join(target_dir, node)
    if os.path.exists(os.path.join(path, ".git")):
        print(f"   ⏭️ Nodo ya clonado: {node}")
        return True
    print(f"   ⬇️ git clone --recursive {node}")
    r = subprocess.run(["git", "clone", "--recursive", raw_url, path], capture_output=True, text=True)
    if r.returncode != 0:
        print(f"      ❌ Error al clonar {node}: {r.stderr.strip()[-300:]}")
        return False
    req = os.path.join(path, "requirements.txt")
    if os.path.exists(req):
        print(f"      📦 Instalando dependencias de {node}...")
        res = subprocess.run(["uv", "pip", "install", "--system", "-r", req], capture_output=True, text=True)
        if res.returncode != 0:
            print(f"      ⚠️ Advertencia: dependencias con error:\n{res.stderr.strip()[-300:]}")
        else:
            print(f"      ✅ Dependencias instaladas con éxito.")
    return True

# =========================================================
#  8. POBLAR COLA SEGÚN SELECCIÓN DE PRESET Y FORMULARIO
# =========================================================
QUEUE_PATH = ("/content/drive/MyDrive/ComfyUI/.download_queue.json"
              if os.path.exists("/content/drive/MyDrive")
              else os.path.join(WORKSPACE, ".download_queue.json"))

# Ejecutar auto-purga preventiva si está activa o si se seleccionó la acción de purga
if AUTO_PURGA_RESIDUOS or ACCION == "🧹 Purgar Residuos y Sanar Caché":
    purgar_residuos_y_sanar_cache(verbose=True)

PENDING = {k: [] for k in DIRS}

# A) Cargar elementos del Preset si fue seleccionado
if WORKFLOW_PRESET in CATALOG and WORKFLOW_PRESET != "Personalizado (Solo URLs manuales)":
    preset = CATALOG[WORKFLOW_PRESET]
    print(f"\n📦 Cargando Preset: {preset.get('title', WORKFLOW_PRESET)}")
    print(f"   ↳ {preset.get('description', '')}")
    for cat, urls in preset.get("models", {}).items():
        if cat in PENDING:
            for u in urls:
                u_clean = clean_url(u)
                if u_clean and u_clean not in PENDING[cat]:
                    PENDING[cat].append(u_clean)
    for nu in preset.get("custom_nodes", []):
        nu = nu.strip()
        if nu and nu not in PENDING["custom_nodes"]:
            PENDING["custom_nodes"].append(nu)

# B) Añadir URLs manuales de los campos del formulario
MANUAL_SEEDS = {
    "diffusion_models":      DIFFUSION_URLS,
    "text_encoders":         TEXT_ENCODER_URLS,
    "checkpoints":           CHECKPOINT_URLS,
    "unet":                  UNET_DIFFUSION_URLS,
    "vae":                   VAE_URLS,
    "loras":                 LORA_URLS,
    "controlnet":            CONTROLNET_URLS,
    "upscale_models":        UPSCALE_MODELS_URLS,
    "custom_nodes":          CUSTOM_NODE_URLS,
}

for cat, raw in MANUAL_SEEDS.items():
    if not raw:
        continue
    for u in raw.replace(',', '\n').split('\n'):
        u_clean = clean_url(u)
        if u_clean and u_clean not in PENDING[cat]:
            PENDING[cat].append(u_clean)

# C) Si se solicitó descargar cola pendiente anterior
if ACCION == "Descargar Cola Pendiente" and os.path.exists(QUEUE_PATH):
    try:
        with open(QUEUE_PATH, "r", encoding="utf-8") as f:
            saved = json.load(f)
        for cat, urls in saved.items():
            if cat in PENDING:
                for u in urls:
                    u_clean = clean_url(u)
                    if u_clean and u_clean not in PENDING[cat]:
                        PENDING[cat].append(u_clean)
    except Exception:
        pass

# D) Guardar como nuevo preset si el usuario especificó un nombre
if GUARDAR_COMO_PRESET.strip():
    new_preset_key = GUARDAR_COMO_PRESET.strip()
    CATALOG[new_preset_key] = {
        "title": f"⭐ {new_preset_key}",
        "description": "Preset personalizado guardado por el usuario.",
        "models": {k: list(v) for k, v in PENDING.items() if k != "custom_nodes" and v},
        "custom_nodes": list(PENDING.get("custom_nodes", []))
    }
    try:
        os.makedirs(os.path.dirname(PRESETS_FILE), exist_ok=True)
        with open(PRESETS_FILE, "w", encoding="utf-8") as f:
            json.dump(CATALOG, f, indent=2, ensure_ascii=False)
        print(f"💾 ¡Nuevo preset guardado exitosamente en {PRESETS_FILE}!: '{new_preset_key}'")
    except Exception as e:
        print(f"⚠️ Error al guardar preset: {e}")

# Guardar cola saneada en disco
try:
    os.makedirs(os.path.dirname(QUEUE_PATH), exist_ok=True)
    with open(QUEUE_PATH, "w", encoding="utf-8") as f:
        json.dump(PENDING, f, indent=2)
except Exception:
    pass

# =========================================================
#  9. RESUMEN Y EJECUCIÓN
# =========================================================
total_items = sum(len(v) for v in PENDING.values())

display(HTML(f'''
<div style="background:linear-gradient(90deg,#1e3c72 0%,#2a5298 100%);color:#fff;padding:12px;border-radius:8px;font-family:sans-serif;margin-bottom:12px;">
  <b style="font-size:15px;">📦 Estado del Downloader: {WORKFLOW_PRESET}</b><br>
  <span style="font-size:12px;color:#d1e0fc;">Acción: <b>{ACCION}</b> | Total en cola: <b>{total_items} elemento(s)</b> | Auto-Purga: <b>{'Activada' if AUTO_PURGA_RESIDUOS else 'Desactivada'}</b></span>
</div>
'''))

if ACCION == "🧹 Purgar Residuos y Sanar Caché":
    print("✅ Operación de purga y sanación completada. Puedes seleccionar 'Descargar e Instalar Todo (1 Clic)' para procesar tu workflow.")
elif total_items == 0:
    print("ℹ️ La cola está vacía. Selecciona un Preset o ingresa URLs en el formulario y presiona Ejecutar.")
else:
    print(f"📋 Elementos preparados para procesamiento ({total_items} total):")
    for cat, urls in PENDING.items():
        if urls:
            print(f"  📁 [{cat}] ({len(urls)} archivo(s)):")
            for i, u in enumerate(urls):
                print(f"     {i+1}. {mask(u)}")

# Ejecutar descarga e instalación si la acción lo indica
if "Descargar e Instalar Todo" in ACCION or "Descargar Cola Pendiente" in ACCION:
    if total_items > 0:
        print("\n" + "="*60)
        print(f"🚀 Iniciando descarga e instalación ({total_items} elemento(s))...")
        print("="*60)
        ok = 0
        fail = 0
        for cat in list(PENDING.keys()):
            urls_to_process = list(PENDING[cat])
            for u in urls_to_process:
                print(f"\n▶ [{cat}] {mask(u)}")
                try:
                    target_dir = DIRS[cat]
                    os.makedirs(target_dir, exist_ok=True)
                    if cat == "custom_nodes":
                        good = download_node(u, target_dir)
                    else:
                        good = download_file(u, target_dir)

                    if good:
                        ok += 1
                        if u in PENDING[cat]:
                            PENDING[cat].remove(u)
                        with open(QUEUE_PATH, "w", encoding="utf-8") as f:
                            json.dump(PENDING, f, indent=2)
                    else:
                        fail += 1
                except Exception as e:
                    print(f"  ❌ Error inesperado: {e}")
                    fail += 1

        print("\n" + "="*60)
        print(f"🏁 Finalizado: ✅ {ok} completadas | ❌ {fail} fallidas | ⏳ {sum(len(v) for v in PENDING.values())} restantes")
        print("="*60)
        if ok > 0:
            display(HTML('<div style="background:#28a745;color:#fff;padding:8px 12px;border-radius:6px;font-weight:bold;text-align:center;margin-top:10px;">🎉 ¡Setup completado! Todos los modelos y nodos están listos en sus carpetas.</div>'))
    else:
        print("⚠️ No hay elementos para descargar.")
elif ACCION == "Solo Poblar la Cola":
    print("\n✅ La cola ha sido poblada y guardada. Cambia la Acción a 'Descargar e Instalar Todo' para iniciar las descargas.")


In [ ]:
#@title 3. Anti-Disconnect Keep-Alive
#@markdown Runs a background audio heartbeat and JavaScript ping to keep the Colab container awake.

from IPython.display import HTML, display

display(HTML('''
<div style="padding: 10px; background: #222; border-left: 4px solid #00d2ff; color: #eee; font-family: sans-serif;">
  <b>🔊 Keep-Alive Protocol Active</b><br>
  <span style="font-size: 12px; color: #aaa;">Browser tab sleep prevention running via web audio loop & activity ping.</span>
  <div style="margin-top: 8px;">
    <audio src="https://raw.githubusercontent.com/anars/blank-audio/master/10-minutes-of-silence.mp3" autoplay loop controls style="height: 30px; width: 250px;"></audio>
  </div>
</div>
<script>
  function keepAlive() {
    console.log("[Keep-Alive] Ping sent: " + new Date().toISOString());
  }
  setInterval(keepAlive, 60000);
</script>
'''))

In [ ]:
#@title 4. Start ComfyUI Session (VRAM-Optimized & Tunnel-Safe)
#@markdown Estrategia de VRAM unificada: reserva + atención + caché. Sin flags duplicados.
#@markdown Detección de CUDA previa y limpieza robusta de procesos/túneles.

import subprocess
import threading
import time
import socket
import os
import sys
import re
import shutil
import urllib.request
from IPython.display import HTML, display
from google.colab import output

# =========================================================
#  PARÁMETROS DE USUARIO
# =========================================================
TUNNEL_MODE = "Cloudflare (TLS bypass)" #@param ["Cloudflare (TLS bypass)", "Localtunnel-Plus (keep-alive)", "Proxy Nativo Colab", "Pinggy (SSH)"]
AUTO_FALLBACK = True #@param {type:"boolean"}

# --- Perfil de memoria (mapea a flags concretos) ---
# Standard: deja que ComfyUI gestione + reserva 1 GB
# T4-safe: reserva 2 GB + split attention (Flux/SD3.5 en T4)
# Low VRAM: lowvram + split attention + cache-none (GPUs pequeñas)
# High VRAM: highvram (A100/40GB+)
MEMORY_PROFILE = "Standard (Auto-Detect)" #@param ["Standard (Auto-Detect)", "T4-safe (16GB + reserve)", "Low VRAM (<10GB)", "High VRAM (A100+)"]

# --- Backend de atención (mutuamente excluyente) ---
ATTENTION_BACKEND = "Default (auto)" #@param ["Default (auto)", "Split (low VRAM)", "PyTorch (safe)", "Sage (fast, NVIDIA)"]

# --- Modo de caché ---
CACHE_MODE = "Default (smart)" #@param ["Default (smart)", "None (free VRAM/RAM)", "Classic"]

# --- VRAM a reservar para el sistema/UI (GB) ---
RESERVE_VRAM_GB = 1 #@param {type:"slider", min:0, max:4, step:0.5}

# --- Frontend ---
FRONTEND_VERSION = "Latest (default)" #@param ["Latest (default)", "1.53.4 (stable)", "1.51.9 (older stable)", "1.49.8 (older)", "Custom..."]
CUSTOM_FRONTEND_VERSION = "" #@param {type:"string"}
CLEAR_FRONTEND_CACHE = True #@param {type:"boolean"}

PORT = 8188
WORKSPACE = "/content/ComfyUI"

# =========================================================
#  1. MATAR PROCESOS FANTASMA (mejorado)
# =========================================================
print("🔄 Liberando puerto 8188 y cerrando túneles previos...")
for pat in ["main.py", "cloudflared", r"\blt\b", "pinggy"]:
    subprocess.run(["pkill", "-9", "-f", pat], stderr=subprocess.DEVNULL)

# fuser puede no estar instalado; fallback con lsof
subprocess.run(["fuser", "-k", f"{PORT}/tcp"], stderr=subprocess.DEVNULL)
try:
    subprocess.run(f"lsof -ti:{PORT} | xargs -r kill -9", shell=True,
                   stderr=subprocess.DEVNULL, timeout=5)
except Exception:
    pass
time.sleep(2)

# =========================================================
#  2. MAPA DE VERSIONES DE FRONTEND
# =========================================================
FRONTEND_MAP = {
    "Latest (default)":            None,
    "1.53.4 (stable)":             "Comfy-Org/ComfyUI_frontend@1.53.4",
    "1.51.9 (older stable)":       "Comfy-Org/ComfyUI_frontend@1.51.9",
    "1.49.8 (older)":              "Comfy-Org/ComfyUI_frontend@1.49.8",
    "Custom...":                   None,
}
if FRONTEND_VERSION == "Custom...":
    v = CUSTOM_FRONTEND_VERSION.strip()
    resolved_frontend = (v if "@" in v else f"Comfy-Org/ComfyUI_frontend@{v}") if v else None
else:
    resolved_frontend = FRONTEND_MAP[FRONTEND_VERSION]

# =========================================================
#  3. LIMPIEZA DE CACHÉ DE FRONTEND
# =========================================================
if CLEAR_FRONTEND_CACHE:
    print("🧹 Limpiando caché de frontend...")
    for path in [os.path.join(WORKSPACE, "web_custom_versions"),
                 os.path.join(WORKSPACE, "web")]:
        if os.path.exists(path):
            shutil.rmtree(path, ignore_errors=True)
            print(f"   ✔ Eliminado: {path}")

display(HTML('<div style="background:linear-gradient(90deg,#ff416c 0%,#ff4b2b 100%);color:#fff;padding:12px;border-radius:6px;font-weight:bold;text-align:center;">[SYSTEM] Booting VRAM-Optimized ComfyUI Engine...</div>'))

# =========================================================
#  4. DIAGNÓSTICO DE CUDA (previo a construir ARGS)
# =========================================================
try:
    import torch
    CUDA_OK = torch.cuda.is_available()
except Exception:
    CUDA_OK = False

if not CUDA_OK:
    print("⚠️ CUDA NO disponible. Los flags de VRAM/atención no aplican.")
    print("   Cambia a Runtime → Change runtime type → T4 GPU.")
    print("   Si el runtime ya es GPU, reinstala torch CUDA:")
    print("      uv pip install --system torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

# =========================================================
#  5. CONSTRUCCIÓN DE ARGS (optimización unificada)
# =========================================================
ARGS = [
    "--listen", "0.0.0.0",
    "--port", str(PORT),
    "--preview-method", "auto",
    "--enable-cors-header", "*",
    "--disable-auto-launch",
]

# --- 5.1 Estrategia de VRAM (mutuamente excluyente) ---
if "Low VRAM" in MEMORY_PROFILE:
    ARGS.append("--lowvram")
    print("📉 Perfil VRAM: --lowvram (GPUs <10GB)")
elif "High VRAM" in MEMORY_PROFILE:
    ARGS.append("--highvram")
    print("📈 Perfil VRAM: --highvram (A100/40GB+)")
elif "T4-safe" in MEMORY_PROFILE:
    # No se usa --lowvram; se confía en reserve + split attention.
    print("🛡️ Perfil VRAM: T4-safe (sin lowvram; reserve + split attention)")
else:
    print("⚖️ Perfil VRAM: Standard (smart offload)")

# --- 5.2 Reserva de VRAM (compatible con cualquier perfil) ---
if RESERVE_VRAM_GB > 0 and CUDA_OK:
    ARGS.extend(["--reserve-vram", str(RESERVE_VRAM_GB)])
    print(f"   ↳ VRAM reservada para sistema/UI: {RESERVE_VRAM_GB} GB")

# --- 5.3 Backend de atención (mutuamente excluyente) ---
if "Split" in ATTENTION_BACKEND:
    ARGS.append("--use-split-cross-attention")
    print("   ↳ Atención: split (reduce VRAM)")
elif "PyTorch" in ATTENTION_BACKEND:
    ARGS.append("--use-pytorch-cross-attention")
    print("   ↳ Atención: PyTorch nativa")
elif "Sage" in ATTENTION_BACKEND:
    ARGS.append("--use-sage-attention")
    print("   ↳ Atención: Sage (rápida, NVIDIA)")

# --- 5.4 Modo de caché ---
if "None" in CACHE_MODE:
    ARGS.append("--cache-none")
    print("   ↳ Caché: none (libera VRAM/RAM entre modelos)")
elif "Classic" in CACHE_MODE:
    ARGS.append("--cache-classic")
    print("   ↳ Caché: classic")

# --- 5.5 cuda-malloc condicional ---
if CUDA_OK:
    ARGS.append("--cuda-malloc")
else:
    # Si no hay CUDA, no pasamos --cuda-malloc para evitar errores.
    pass

# --- 5.6 Frontend ---
if resolved_frontend:
    ARGS.extend(["--front-end-version", resolved_frontend])
    print(f"   ↳ Frontend fijado: {resolved_frontend}")
else:
    print("   ↳ Frontend: default (pip package)")

# --- 5.7 Log de flags finales ---
print(f"\n[SYSTEM] Flags efectivos: {' '.join(ARGS)}")
print("[SYSTEM] Logs:\n" + "-" * 60)

# =========================================================
#  6. UTILIDADES (sin cambios)
# =========================================================
def drain_stream(stream):
    try:
        for _ in iter(stream.readline, ""):
            pass
    except Exception:
        pass

def wait_for_port(port, timeout=180):
    start = time.time()
    while time.time() - start < timeout:
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        res = s.connect_ex(("127.0.0.1", port))
        s.close()
        if res == 0:
            print(f"✅ Puerto {port} LISTENING ({time.time()-start:.1f}s).")
            return True
        time.sleep(1.0)
    print(f"❌ Puerto {port} no abrió en {timeout}s.")
    return False

def http_probe(url, timeout=10):
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0 ComfyProbe"})
        with urllib.request.urlopen(req, timeout=timeout) as r:
            return 200 <= r.status < 400
    except Exception:
        return False

def wait_for_tunnel_ready(url, timeout=45):
    print(f"🔎 Verificando túnel ({url})...")
    start = time.time()
    while time.time() - start < timeout:
        if http_probe(url):
            print(f"✅ Túnel OK ({time.time()-start:.1f}s).")
            return True
        time.sleep(2)
    print(f"⚠️ Túnel tardó >{timeout}s; prueba abrir el enlace igualmente.")
    return False

def render_banner(name, url, note=""):
    display(HTML(f'''
    <div style="background:linear-gradient(90deg,#11998e,#38ef7d);color:#000;padding:14px;border-radius:8px;font-weight:bold;font-size:15px;margin:15px 0;text-align:center;">
      🚀 <b>{name} READY:</b>
      <a href="{url}" target="_blank" style="color:#002b49;text-decoration:underline;margin-left:8px;">{url}</a>
      {f'<br><span style="font-size:12px;color:#003b1f;">{note}</span>' if note else ''}
    </div>
    '''))

# =========================================================
#  7. LANZADORES DE TÚNELES (sin cambios)
# =========================================================
def launch_cloudflare(port):
    if not os.path.exists("/usr/local/bin/cloudflared"):
        print("📦 Descargando cloudflared...")
        subprocess.run(["wget", "-q", "-nc",
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
            "-O", "/usr/local/bin/cloudflared"])
        subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"])
    cmd = ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{port}",
           "--no-autoupdate", "--edge-ip-version", "4", "--protocol", "http2"]
    print("🌩️ Cloudflare (http2)...")
    p = subprocess.Popen(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True, bufsize=1)
    url, start = None, time.time()
    for line in iter(p.stderr.readline, ""):
        m = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
        if m:
            url = m.group(0); break
        if time.time() - start > 45: break
    threading.Thread(target=drain_stream, args=(p.stderr,), daemon=True).start()
    if not url:
        print("❌ Cloudflare sin URL en 45s."); return None
    render_banner("CLOUDFLARE TUNNEL", url, "HTTP/2 activo.")
    wait_for_tunnel_ready(url, 45)
    return url

def launch_localtunnel_plus(port):
    print("📦 localtunnel-plus...")
    os.system("npm install -g localtunnel-plus >/dev/null 2>&1")
    try:
        ip = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=5).read().decode().strip()
    except Exception:
        ip = "N/A"
    p = subprocess.Popen(["lt", "--port", str(port), "--keep-alive"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url, start = None, time.time()
    for line in iter(p.stdout.readline, ""):
        m = re.search(r"https://[a-z0-9-]+\.loca\.lt", line)
        if m:
            url = m.group(0); break
        if time.time() - start > 30: break
    threading.Thread(target=drain_stream, args=(p.stdout,), daemon=True).start()
    if not url:
        print("❌ Localtunnel-Plus sin URL."); return None
    render_banner("LOCALTUNNEL-PLUS", url, f"Password: {ip}")
    wait_for_tunnel_ready(url, 30)
    return url

def launch_native_colab(port):
    url = output.eval_js(f"google.colab.kernel.proxyPort({port})")
    render_banner("PROXY NATIVO COLAB", url)
    return url

def launch_pinggy(port):
    p = subprocess.Popen(["ssh", "-o", "StrictHostKeyChecking=no", "-p", "443",
                          f"-R0:localhost:{port}", "a.pinggy.io"],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    url, start = None, time.time()
    for line in iter(p.stdout.readline, ""):
        m = re.search(r"https://[a-zA-Z0-9-]+\.pinggy\.link", line)
        if m:
            url = m.group(0); break
        if time.time() - start > 30: break
    threading.Thread(target=drain_stream, args=(p.stdout,), daemon=True).start()
    if not url:
        print("❌ Pinggy sin URL."); return None
    render_banner("PINGGY TUNNEL", url)
    return url

# =========================================================
#  8. ORQUESTADOR CON FALLBACK
# =========================================================
def tunnel_worker(mode, port):
    if not wait_for_port(port, timeout=180):
        print("❌ ComfyUI no escucha; abortando túnel.")
        return
    url = None
    if "Cloudflare" in mode:       url = launch_cloudflare(port)
    elif "Localtunnel-Plus" in mode: url = launch_localtunnel_plus(port)
    elif "Proxy Nativo" in mode:   url = launch_native_colab(port)
    elif "Pinggy" in mode:         url = launch_pinggy(port)
    if not url and AUTO_FALLBACK:
        print("\n🔁 Fallback: Proxy Nativo de Colab...\n")
        url = launch_native_colab(port)
    print(f"\n🌐 URL final: {url}" if url else "\n❌ Sin túnel.")

threading.Thread(target=tunnel_worker, daemon=True, args=(TUNNEL_MODE, PORT)).start()

# =========================================================
#  9. ARRANCAR COMFYUI
# =========================================================
if os.path.exists(os.path.join(WORKSPACE, "main.py")):
    os.chdir(WORKSPACE)
    print(f"[SYSTEM] ComfyUI | Perfil: {MEMORY_PROFILE} | Túnel: {TUNNEL_MODE}")
    cmd = ["python", "main.py"] + ARGS
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    for line in iter(proc.stdout.readline, ""):
        sys.stdout.write(line); sys.stdout.flush()
else:
    print("❌ [FATAL] main.py no encontrado. Ejecuta primero la Celda 1.")

🔄 Liberando puerto 8188 y cerrando túneles previos...
🧹 Limpiando caché de frontend...


⚖️ Perfil VRAM: Standard (smart offload)
   ↳ VRAM reservada para sistema/UI: 1 GB
   ↳ Frontend: default (pip package)

[SYSTEM] Flags efectivos: --listen 0.0.0.0 --port 8188 --preview-method auto --enable-cors-header * --disable-auto-launch --reserve-vram 1 --cuda-malloc
[SYSTEM] Logs:
------------------------------------------------------------
[SYSTEM] ComfyUI | Perfil: Standard (Auto-Detect) | Túnel: Cloudflare (TLS bypass)
[INFO] setup plugin alembic.autogenerate.schemas
[INFO] setup plugin alembic.autogenerate.tables
[INFO] setup plugin alembic.autogenerate.types
[INFO] setup plugin alembic.autogenerate.constraints
[INFO] setup plugin alembic.autogenerate.defaults
[INFO] setup plugin alembic.autogenerate.comments
[INFO] setup plugin alembic.ext.checkconstraint_byname
[START] Security scan
[DONE] Security scan
## ComfyUI-Manager: installing dependencies done.
** ComfyUI startup time: 2026-11-03 21:10:38.587
** Platform: Linux
** Python version: 3.13.16 (main, Oct  1 2026, 15:40:2

🔎 Verificando túnel (https://restricted-coverage-triangle-supervisor.trycloudflare.com)...
✅ Túnel OK (30.8s).

🌐 URL final: https://restricted-coverage-triangle-supervisor.trycloudflare.com
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /extensions/core/groupNode.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] Detected import of deprecated legacy API: /scripts/ui/components/buttonGroup.js. This is likely caused by a custom node extension using outdated APIs. Please update your extensions or contact the extension author for an updated version.
[WARNING] [DEPRECATION WARNING] D

KeyboardInterrupt: 

In [ ]:
#@title 5. High-Speed Workspace Backup to Drive
#@markdown Archives custom nodes, user workflows, and settings directly to Google Drive using multi-threaded compression.

import os
WORKSPACE = "/content/ComfyUI"
DRIVE_WORKSPACE = "/content/drive/MyDrive/ComfyUI"
CACHE_TAR = os.path.join(DRIVE_WORKSPACE, "comfy_ui_cache.tar")

if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount('/content/drive')

os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

if os.path.exists(WORKSPACE):
    print("🔄 Archiving custom nodes & user directory with pigz...")
    os.chdir(WORKSPACE)
    os.system(f"tar -I pigz -cf '{CACHE_TAR}' custom_nodes user")
    print(f"✅ Fast archive saved to Drive: {CACHE_TAR}")
else:
    print("⚠️ Workspace not found.")